# Apoio para organizar dataset de frutas

Este notebook é apenas um material de apoio para organizar e conferir imagens do dataset.

Ele **não** faz segmentação, não extrai features e não treina classificadores.

Use para:
- conferir se as imagens estão nas pastas certas;
- contar imagens por classe;
- copiar uma quantidade balanceada por classe;
- gerar um CSV simples de conferência;
- visualizar algumas amostras do dataset.


## Estrutura esperada das pastas

Coloque as imagens assim:

```text
dataset_raw/
├── fresh/
└── rotten/
```

Ou, se quiser usar português:

```text
dataset_raw/
├── saudavel/
└── podre/
```


In [ ]:
# Instale esta biblioteca se ainda não tiver:
# !pip install pillow

from pathlib import Path
import csv
import random
import shutil

from PIL import Image, ImageOps, ImageDraw


## Configuração inicial

Altere os nomes das classes se suas pastas tiverem outro nome.

Exemplo:
- `classes = ["fresh", "rotten"]`
- ou `classes = ["saudavel", "podre"]`


In [ ]:
entrada = Path("dataset_raw")
saida = Path("dataset_trabalho")
pasta_outputs = Path("outputs")

# Troque para ["saudavel", "podre"] se suas pastas estiverem em português
classes = ["fresh", "rotten"]

# Quantidade máxima de imagens por classe.
# Use None para copiar todas as imagens encontradas.
max_por_classe = 100

# Semente para deixar o sorteio reprodutível
seed = 42

# Extensões aceitas
extensoes_validas = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


## Criar pastas base

Rode esta célula para criar a estrutura caso ela ainda não exista.

In [ ]:
for nome_classe in classes:
    (entrada / nome_classe).mkdir(parents=True, exist_ok=True)

saida.mkdir(parents=True, exist_ok=True)
pasta_outputs.mkdir(parents=True, exist_ok=True)

print("Pastas criadas/conferidas com sucesso.")
print(f"Coloque suas imagens dentro de: {entrada.resolve()}")


## Função para listar imagens

Esta função procura arquivos de imagem dentro de uma pasta de classe.

In [ ]:
def listar_imagens(pasta: Path) -> list[Path]:
    """Retorna uma lista ordenada com imagens válidas de uma pasta."""
    if not pasta.exists():
        raise FileNotFoundError(f"Pasta não encontrada: {pasta}")

    imagens = [
        caminho
        for caminho in pasta.iterdir()
        if caminho.is_file() and caminho.suffix.lower() in extensoes_validas
    ]

    return sorted(imagens)


## Conferir quantidade de imagens por classe

Antes de copiar, confira se as imagens realmente estão dentro das pastas corretas.

In [ ]:
for nome_classe in classes:
    pasta_classe = entrada / nome_classe
    imagens = listar_imagens(pasta_classe)
    print(f"Classe '{nome_classe}': {len(imagens)} imagem(ns) encontrada(s).")


## Copiar dataset balanceado

Esta célula copia até `max_por_classe` imagens de cada classe para a pasta `dataset_trabalho`.

Exemplo: se `max_por_classe = 100`, ele tenta copiar 100 imagens de cada classe.

In [ ]:
def copiar_dataset_balanceado(
    entrada: Path,
    saida: Path,
    classes: list[str],
    max_por_classe: int | None,
    seed: int,
) -> list[dict]:
    """Copia imagens para uma pasta de trabalho e retorna linhas para um CSV."""
    random.seed(seed)
    saida.mkdir(parents=True, exist_ok=True)

    linhas = []

    for rotulo_numerico, nome_classe in enumerate(classes):
        pasta_origem = entrada / nome_classe
        imagens = listar_imagens(pasta_origem)

        if not imagens:
            print(f"[AVISO] Nenhuma imagem encontrada em: {pasta_origem}")
            continue

        random.shuffle(imagens)

        if max_por_classe is not None:
            imagens = imagens[:max_por_classe]

        pasta_destino = saida / nome_classe
        pasta_destino.mkdir(parents=True, exist_ok=True)

        print(f"Classe '{nome_classe}': copiando {len(imagens)} imagem(ns).")

        for indice, origem_img in enumerate(imagens, start=1):
            novo_nome = f"{nome_classe}_{indice:04d}{origem_img.suffix.lower()}"
            destino_img = pasta_destino / novo_nome

            shutil.copy2(origem_img, destino_img)

            try:
                with Image.open(destino_img) as img:
                    largura, altura = img.size
            except Exception:
                largura, altura = None, None

            linhas.append(
                {
                    "caminho": str(destino_img.as_posix()),
                    "classe": nome_classe,
                    "rotulo": rotulo_numerico,
                    "largura": largura,
                    "altura": altura,
                }
            )

    return linhas


linhas = copiar_dataset_balanceado(
    entrada=entrada,
    saida=saida,
    classes=classes,
    max_por_classe=max_por_classe,
    seed=seed,
)


## Salvar CSV de conferência

Este CSV ainda não é o `X.csv` final do projeto. Ele serve apenas para registrar quais imagens foram copiadas e quais são seus rótulos.

In [ ]:
def salvar_manifesto(saida: Path, linhas: list[dict]) -> Path:
    """Salva um CSV com informações básicas das imagens."""
    caminho_csv = saida / "dataset_manifest.csv"

    with caminho_csv.open("w", newline="", encoding="utf-8") as arquivo:
        campos = ["caminho", "classe", "rotulo", "largura", "altura"]
        escritor = csv.DictWriter(arquivo, fieldnames=campos)
        escritor.writeheader()
        escritor.writerows(linhas)

    return caminho_csv


caminho_csv = salvar_manifesto(saida, linhas)
print(f"CSV salvo em: {caminho_csv}")


## Mostrar resumo final

Aqui você confere se a quantidade ficou balanceada entre as classes.

In [ ]:
print("Resumo do dataset selecionado:")
total = 0

for nome_classe in classes:
    qtd = sum(1 for linha in linhas if linha["classe"] == nome_classe)
    total += qtd
    print(f"- {nome_classe}: {qtd} imagem(ns)")

print(f"Total: {total} imagem(ns)")


## Criar grade visual com amostras

Esta parte gera uma imagem com algumas amostras de cada classe, para conferir visualmente se as pastas estão corretas.

In [ ]:
def criar_grade_amostras(
    linhas: list[dict],
    classes: list[str],
    caminho_saida: Path,
    tamanho: int = 160,
    por_classe: int = 6,
) -> None:
    """Cria uma grade simples com algumas imagens de cada classe."""
    caminho_saida.parent.mkdir(parents=True, exist_ok=True)

    selecionadas = []
    for nome_classe in classes:
        imgs_classe = [linha for linha in linhas if linha["classe"] == nome_classe]
        selecionadas.extend(imgs_classe[:por_classe])

    if not selecionadas:
        print("[AVISO] Não há imagens para criar a grade de amostras.")
        return

    colunas = por_classe
    linhas_grade = len(classes)
    largura_total = colunas * tamanho
    altura_total = linhas_grade * (tamanho + 30)

    grade = Image.new("RGB", (largura_total, altura_total), "white")
    desenho = ImageDraw.Draw(grade)

    for linha_idx, nome_classe in enumerate(classes):
        imgs_classe = [x for x in selecionadas if x["classe"] == nome_classe]

        for coluna_idx, info in enumerate(imgs_classe):
            caminho_img = Path(info["caminho"])

            try:
                with Image.open(caminho_img) as img:
                    img = ImageOps.exif_transpose(img).convert("RGB")
                    img.thumbnail((tamanho, tamanho))

                    x = coluna_idx * tamanho + (tamanho - img.width) // 2
                    y = linha_idx * (tamanho + 30) + 20 + (tamanho - img.height) // 2
                    grade.paste(img, (x, y))

            except Exception as erro:
                print(f"[AVISO] Erro ao abrir {caminho_img}: {erro}")

        y_texto = linha_idx * (tamanho + 30) + 2
        desenho.text((5, y_texto), nome_classe, fill="black")

    grade.save(caminho_saida)
    print(f"Grade de amostras salva em: {caminho_saida}")


caminho_grade = pasta_outputs / "amostras_dataset.jpg"
criar_grade_amostras(
    linhas=linhas,
    classes=classes,
    caminho_saida=caminho_grade,
)


## Visualizar a grade dentro do notebook

Se a imagem foi gerada corretamente, ela aparecerá abaixo.

In [ ]:
from IPython.display import display

if caminho_grade.exists():
    display(Image.open(caminho_grade))
else:
    print("A grade ainda não foi criada.")


## Próximo passo

Depois desta etapa, a equipe deve desenvolver por conta própria o pipeline principal:

1. segmentação ou isolamento do objeto;
2. extração de features manuais;
3. montagem de X e y;
4. treinamento de classificadores clássicos;
5. avaliação com métricas e matriz de confusão.
